[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb)

# Lab 2: Trace Your Agent

### Instrument the agent with OpenTelemetry and inspect its traces in Phoenix

In notebook 1, you built the agent. Here, the OpenAI instrumentor captures LLM calls
automatically, while manual spans make the agent and its tool flow visible. You will
inspect the full trace—including tool inputs and outputs—in Phoenix.

The workshop's canonical agent remains in `utils.py`; this notebook attaches a tracer to
that implementation rather than creating a second agent that could drift.


---
## 1. Install Phoenix tracing libraries

The observability stack uses Phoenix's OpenTelemetry registration and OpenInference's
OpenAI instrumentor. This pins the local Phoenix package to the same version as the
workshop's Phoenix board.


In [ ]:
%pip install -q arize-phoenix==22.0.0 openinference-instrumentation-openai==0.1.76

> Colab may report resolver warnings about packages already installed in its runtime.
> If an import or instrumentation error follows, restart the runtime and rerun from the top.
> This install runs before the agent and OpenTelemetry imports.


---
## Bootstrap the workshop agent

Fetch `utils.py` only if needed, load the synthetic ledger, and connect through the same
workshop gateway and seat key as notebook 1.


In [ ]:
import urllib.request
import os

if not os.path.exists("utils.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils

utils.load_data()
utils.connect()
SEAT = utils.seat_tag()
from utils import NAIVE_SYSTEM_PROMPT, run_agent

print("bootstrap done | seat tag:", SEAT)


## 2. Start Phoenix and register an OpenTelemetry provider

For this workshop, Phoenix runs locally in the notebook runtime. `register` configures
its trace exporter; the endpoint is the collector URL, not the Phoenix UI URL.


In [ ]:
import phoenix as px
from phoenix.otel import register

session = px.launch_app()
print("Phoenix UI:", session.url)


In [ ]:
tracer_provider = register(
    project_name=f"workshop-{SEAT}",
    endpoint="http://127.0.0.1:6006/v1/traces",
    protocol="http/protobuf",
    batch=False,
)
print("OpenTelemetry collector:", "http://127.0.0.1:6006/v1/traces")


## 3. Instrument LLM calls and attach the agent tracer

`OpenAIInstrumentor` creates an LLM span for each OpenAI SDK call. The canonical agent
adds the surrounding spans: an **AGENT** root, a **CHAIN** for each router call, a nested
LLM span, then a tool-handling **CHAIN** containing one **TOOL** span per tool execution.
Inputs, outputs, and tool errors are recorded as span attributes.


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(tracer_provider.get_tracer(__name__))
print("OpenAI calls instrumented; manual agent/chain/tool spans enabled")


> **Trace data note:** spans contain prompts and tool inputs/outputs. This workshop uses
> synthetic transactions; do not send personal or otherwise sensitive data to a tracer.


---
## 4. Trace a request the ledger may not support

This request asks for a merchant-location breakdown, but the ledger has no location field.
Run the agent, then use the trace to see which steps it took and what the tool returned.
The exact model response can vary.


In [ ]:
missing_location = run_agent(
    "Show me my spending broken down by merchant location.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/missing-location",
)
print(missing_location["answer"])


### Read the trace in Phoenix

Open the Phoenix UI and select project `workshop-<SEAT>`. Expand the trace and follow the
parent-child spans. Find the router's LLM call, the selected tool, and the tool's input
and output. A failed tool call should be visible in its TOOL span rather than disappearing
inside a final answer.


In [ ]:
from phoenix.client import Client

phoenix_client = Client()
spans_df = phoenix_client.spans.get_spans_dataframe(project_identifier=f"workshop-{SEAT}")
trace_columns = [
    column for column in (
        "name", "span_kind", "parent_id", "status_code",
        "attributes.input.value", "attributes.output.value",
        "attributes.llm.token_count.prompt", "attributes.llm.token_count.completion",
    )
    if column in spans_df.columns
]
display(spans_df[trace_columns].tail(12))


---
## 5. Compare with a successful analysis

Run a second request that asks for net expenses after refunds. In Phoenix, compare its
agent → router/LLM → tool span tree with the missing-information example. Notice how the
same instrumentation captures both paths without changing the tracing code.


In [ ]:
net_expenses = run_agent(
    "Use analyze_transactions to calculate net expenses in October after transaction refunds, excluding income.",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/net-expenses-after-refunds",
)
print(net_expenses["answer"])


In [ ]:
spans_df = phoenix_client.spans.get_spans_dataframe(project_identifier=f"workshop-{SEAT}")
display(spans_df[trace_columns].tail(20))


---
## Optional - Publish traces to the room board

This step is optional. It sends your local spans to the shared Phoenix board.


First, remap span IDs so traces from different participants do not collide.


In [ ]:
import copy
import hashlib

def remap_span_ids(spans, seat):
    """Make span and parent IDs unique to this participant."""
    remapped = []
    ids = {}
    for span in spans:
        item = copy.deepcopy(span)
        context = item.get("context") or {}
        for key in ("trace_id", "span_id"):
            old = context.get(key)
            if old:
                context[key] = ids.setdefault(old, hashlib.md5(f"{seat}:{old}".encode()).hexdigest()[:len(old)])
        old_parent = item.get("parent_id")
        if old_parent:
            item["parent_id"] = ids.setdefault(old_parent, hashlib.md5(f"{seat}:{old_parent}".encode()).hexdigest()[:len(old_parent)])
        item.pop("id", None)
        remapped.append(item)
    return remapped


The board URL is fixed for this workshop. Keep its API key in a Colab secret or
environment variable; the publish cell reads it without exposing it in the notebook.


In [ ]:
from phoenix.client import Client

board_url = "https://fmlds-eval-workshop-phoenix.fly.dev"
board_key = utils.get_secret("PHOENIX_API_KEY")
if not board_key:
    print("Add PHOENIX_API_KEY to your workshop secrets first.")
else:
    spans = Client().spans.get_spans(project_identifier=f"workshop-{SEAT}", limit=2000)
    board = Client(base_url=board_url, api_key=board_key or None)
    result = board.spans.log_spans(
        project_identifier=f"seat-{SEAT}", spans=remap_span_ids(spans, SEAT),
    )
    print("published:", result)


---
### Continue the workshop

[Next: Notebook 3 — Evaluations](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)
